# AccentSense — UK Regional Accent Classifier (Local DirectML Edition)
**WavLM Base+ · Attentive Statistics Pooling · 6 UK Dialect Classes**
VIT Bhopal University · AI/ML Capstone Research
---
**Classes (6):** `RP` · `Scottish` · `Welsh` · `Northern` · `West_Midlands` · `Irish`
**Hardware:** Optimized for **AMD Radeon 860M (DirectML / DirectX 12)** in VS Code (`Backend/.venv` Python 3.11)

> **VS Code Kernel Selection:** Click **Select Kernel** (top-right of VS Code) → choose **`Python 3.11 (AccentSense DirectML)`** or **`Backend/.venv/Scripts/python.exe`**.

**Steps:**
1. Verify AMD Radeon 860M DirectML GPU & environment
2. Verify / install dependencies into active kernel
3. Set working directory to `Backend/`
4. Check local VCTK or enable Hugging Face streaming
5. Add Irish English supplement (`ylacombe/english_dialects`)
6. Build speaker-disjoint 6-class manifests
7. Train WavLM (DirectML-optimized: `batch_size=2`, `grad_accum=4`, `num_workers=0`)
8. Evaluate on test set — classification report + confusion matrix
9. Verify local checkpoint for FastAPI backend


In [ ]:
# Step 1: Verify GPU (AMD Radeon 860M via DirectML)
import sys, os, torch

print(f'Python:  {sys.version.split()[0]} ({sys.executable})')
print(f'PyTorch: {torch.__version__}')

DEVICE = None
try:
    import torch_directml
    if torch_directml.is_available():
        DEVICE = torch_directml.device()
        dml_name = torch_directml.device_name(0) if hasattr(torch_directml, 'device_name') else 'DirectX 12 GPU (AMD Radeon 860M)'
        print(f'✓ DirectML Available: True')
        print(f'✓ Active GPU Device:  {DEVICE} ({dml_name})')
except ImportError:
    print('⚠ torch-directml not found in current kernel. Run Step 2 or select the Backend/.venv (Python 3.11) kernel.')

if DEVICE is None:
    if torch.cuda.is_available():
        DEVICE = torch.device('cuda')
        print(f'✓ CUDA GPU: {torch.cuda.get_device_name(0)}')
    else:
        DEVICE = torch.device('cpu')
        print(f'⚠ Falling back to CPU ({os.cpu_count()} threads). Select Python 3.11 (.venv) kernel for DirectML.')


In [ ]:
# Step 2: Verify dependencies in the active VS Code kernel
import subprocess, sys

REQUIRED = [
    ('torch_directml', 'torch-directml'),
    ('torchaudio', 'torchaudio==2.4.1'),
    ('transformers', 'transformers==4.46.3'),
    ('datasets', 'datasets'),
    ('soundfile', 'soundfile'),
    ('sklearn', 'scikit-learn'),
    ('pandas', 'pandas'),
    ('tqdm', 'tqdm'),
]

missing = []
for mod_name, pkg_name in REQUIRED:
    try:
        __import__(mod_name)
    except ImportError:
        missing.append(pkg_name)

if missing:
    print(f'Installing missing packages into active kernel: {missing}')
    subprocess.check_call(['uv', 'pip', 'install', '--python', sys.executable] + missing)
else:
    print('✓ All required packages are installed in this kernel.')


In [ ]:
# Step 3: Resolve local Backend directory & configure Python path
import os, sys

cwd = os.path.abspath(os.getcwd())
if os.path.basename(cwd) == 'notebooks' and os.path.basename(os.path.dirname(cwd)) == 'Backend':
    BACKEND_DIR = os.path.dirname(cwd)
elif os.path.basename(cwd) == 'Backend':
    BACKEND_DIR = cwd
elif os.path.isdir(os.path.join(cwd, 'Backend')):
    BACKEND_DIR = os.path.join(cwd, 'Backend')
else:
    BACKEND_DIR = cwd

os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print('✓ Working directory:', os.getcwd())
print('✓ src module found: ', os.path.isdir(os.path.join(BACKEND_DIR, 'src')))


In [ ]:
# Step 4: Configure VCTK Data Source (Local or Hugging Face Stream)
# Avoids downloading the 11 GB VCTK zip on your laptop by streaming only needed samples from Hugging Face.
import os, glob

os.makedirs('data/vctk', exist_ok=True)
os.makedirs('data/vctk_hf', exist_ok=True)

local_info = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)
cached_hf_wavs = glob.glob('data/vctk_hf/*.wav')

if local_info:
    USE_HF_VCTK = False
    print(f'✓ Found local VCTK dataset: {local_info[0]}')
elif len(cached_hf_wavs) > 100:
    USE_HF_VCTK = True
    print(f'✓ Found {len(cached_hf_wavs)} cached WAV files in data/vctk_hf/')
else:
    USE_HF_VCTK = True
    print('✓ Local VCTK zip not present — Step 6 will stream UK dialect audio directly from Hugging Face.')


In [ ]:
# Step 5: Add Irish English speech supplement
# Uses open-access ylacombe/english_dialects (no token required) and caches WAVs in data/cv_irish/
import os, glob
import soundfile as sf
import numpy as np

os.makedirs('data/cv_irish', exist_ok=True)
existing_irish = sorted(glob.glob('data/cv_irish/*.wav'))
cv_extras = []

if len(existing_irish) >= 50:
    print(f'✓ Using {len(existing_irish)} cached Irish WAV files from data/cv_irish/')
    for i, p in enumerate(existing_irish):
        cv_extras.append({
            'path': p,
            'label': 'Irish',
            'speaker': f'irish_spk_{i % 10}',
            'class_idx': 5
        })
else:
    try:
        from datasets import load_dataset
        irish = []
        print('Streaming Irish English from ylacombe/english_dialects (no auth required)...')
        for subset in ['irish_male', 'irish_female']:
            try:
                ds = load_dataset('ylacombe/english_dialects', subset, split='train', streaming=True)
                for s in ds:
                    irish.append({
                        'audio': s['audio'],
                        'speaker': str(s.get('speaker_id', f'spk_{subset}'))
                    })
                    if len(irish) >= 500:
                        break
                print(f'  ✓ {subset}: collected {len(irish)} clips total')
                if len(irish) >= 500:
                    break
            except Exception as e:
                print(f'  {subset} warning: {e}')

        for i, item in enumerate(irish):
            p = f'data/cv_irish/irish_{i:04d}.wav'
            sf.write(p, np.array(item['audio']['array'], dtype=np.float32), item['audio']['sampling_rate'])
            cv_extras.append({
                'path': p,
                'label': 'Irish',
                'speaker': item['speaker'],
                'class_idx': 5
            })
        print(f'✓ Saved {len(cv_extras)} Irish WAV files to data/cv_irish/')
    except Exception as e:
        print(f'Irish supplement error: {e}')


In [ ]:
# Step 6: Build speaker-disjoint 6-class manifests (Local VCTK or Hugging Face Fallback)
from collections import Counter
import glob, os, sys, json, random
import numpy as np
import soundfile as sf

CLASSES = ('RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands', 'Irish')

VCTK_ACCENT_MAP = {
    'English': 'RP', 'Southern England': 'RP', 'Surrey': 'RP', 'Kent': 'RP',
    'Hampshire': 'RP', 'Essex': 'RP', 'Berkshire': 'RP', 'Oxfordshire': 'RP',
    'London': 'RP', 'East London': 'RP', 'British': 'RP',
    'Scottish': 'Scottish', 'Scotland': 'Scottish', 'Edinburgh': 'Scottish', 'Glasgow': 'Scottish',
    'Welsh': 'Welsh', 'Wales': 'Welsh',
    'Northern England': 'Northern', 'Yorkshire': 'Northern', 'Manchester': 'Northern',
    'Newcastle': 'Northern', 'Geordie': 'Northern', 'Lancashire': 'Northern', 'Leeds': 'Northern',
    'West Midlands': 'West_Midlands', 'Birmingham': 'West_Midlands', 'Midlands': 'West_Midlands',
    'Irish': 'Irish', 'Ireland': 'Irish', 'Dublin': 'Irish',
    'Northern Ireland': 'Irish', 'NorthernIrish': 'Irish',
}

def save_manifest(manifest, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

def speaker_disjoint_splits(manifest, train_ratio=0.75, val_ratio=0.15, seed=42):
    random.seed(seed)
    class_to_speakers = {}
    for entry in manifest:
        cls, spk = entry['label'], entry['speaker']
        class_to_speakers.setdefault(cls, [])
        if spk not in class_to_speakers[cls]:
            class_to_speakers[cls].append(spk)
    train_speakers, val_speakers, test_speakers = set(), set(), set()
    for cls, speakers in class_to_speakers.items():
        random.shuffle(speakers)
        n = len(speakers)
        if n >= 3:
            n_train = max(1, int(n * train_ratio))
            n_val = max(1, int(n * val_ratio))
            train_speakers.update(speakers[:n_train])
            val_speakers.update(speakers[n_train:n_train + n_val])
            test_speakers.update(speakers[n_train + n_val:])
        else:
            # If a dialect subset has fewer than 3 speakers, split clips directly so all splits have representation
            train_speakers.update(speakers)
    train, val, test = [], [], []
    for cls in CLASSES:
        cls_entries = [e for e in manifest if e['label'] == cls]
        spks = class_to_speakers.get(cls, [])
        if len(spks) >= 3:
            train.extend([e for e in cls_entries if e['speaker'] in train_speakers])
            val.extend([e for e in cls_entries if e['speaker'] in val_speakers])
            test.extend([e for e in cls_entries if e['speaker'] in test_speakers])
        elif cls_entries:
            random.shuffle(cls_entries)
            nt = int(len(cls_entries) * train_ratio)
            nv = int(len(cls_entries) * val_ratio)
            train.extend(cls_entries[:nt])
            val.extend(cls_entries[nt:nt + nv])
            test.extend(cls_entries[nt + nv:])
    return train, val, test

INFO_FILES = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)
use_local = (not globals().get('USE_HF_VCTK', False)) and len(INFO_FILES) > 0
manifest = []
speaker_counts = Counter()

# PATH A: Local VCTK extraction available
if use_local:
    print('=== Building manifest from local VCTK ===')
    INFO = INFO_FILES[0]
    WAVD = None
    for pattern in ['data/vctk/wav48_silence_trimmed', 'data/vctk/wav48', 'data/vctk/wav16']:
        if os.path.isdir(pattern):
            WAVD = pattern
            break
    if not WAVD:
        use_local = False

if use_local:
    speaker_accents = {}
    with open(INFO, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('ID'):
                continue
            parts = line.split()
            if len(parts) >= 4:
                speaker_id = parts[0]
                accent = ' '.join(parts[3:]) if len(parts) > 4 else parts[3]
                speaker_accents[speaker_id] = accent
    for speaker_dir in sorted(os.listdir(WAVD)):
        speaker_path = os.path.join(WAVD, speaker_dir)
        if not os.path.isdir(speaker_path) or speaker_dir not in speaker_accents:
            continue
        accent = speaker_accents[speaker_dir]
        class_label = VCTK_ACCENT_MAP.get(accent)
        if class_label is None:
            for k, v in VCTK_ACCENT_MAP.items():
                if k.lower() in accent.lower():
                    class_label = v
                    break
        if class_label is None or class_label not in CLASSES:
            continue
        class_idx = CLASSES.index(class_label)
        audio_files = []
        for ext in ['*.wav', '*.WAV', '*.flac', '*.FLAC']:
            audio_files.extend(glob.glob(os.path.join(speaker_path, ext)))
        for audio_path in audio_files:
            manifest.append({
                'path': audio_path,
                'label': class_label,
                'speaker': speaker_dir,
                'class_idx': class_idx
            })
            speaker_counts[class_label] += 1

# PATH B: Stream from Hugging Face if local VCTK not present
if not use_local or len(manifest) == 0:
    print('\n=== Streaming & building dataset from Hugging Face ===')
    from datasets import load_dataset
    os.makedirs('data/vctk_hf', exist_ok=True)
    hf_cache_meta = 'data/vctk_hf/metadata_cache.json'
    if os.path.exists(hf_cache_meta):
        with open(hf_cache_meta, 'r', encoding='utf-8') as f:
            cached_items = json.load(f)
        for item in cached_items:
            if os.path.exists(item['path']) and item['label'] in CLASSES:
                item['class_idx'] = CLASSES.index(item['label'])
                manifest.append(item)
                speaker_counts[item['label']] += 1
        print(f'✓ Loaded {len(manifest)} cached HF audio entries from {hf_cache_meta}')

    if len(manifest) < 100:
        DIALECT_MAP = {
            'southern_male': 'RP', 'southern_female': 'RP',
            'scottish_male': 'Scottish', 'scottish_female': 'Scottish',
            'welsh_male': 'Welsh', 'welsh_female': 'Welsh',
            'northern_male': 'Northern', 'northern_female': 'Northern',
            'midlands_male': 'West_Midlands', 'midlands_female': 'West_Midlands',
        }
        for subset, cls in DIALECT_MAP.items():
            try:
                ds = load_dataset('ylacombe/english_dialects', subset, split='train', streaming=True)
                cnt = 0
                for idx, row in enumerate(ds):
                    spk = str(row.get('speaker_id', f'spk_{subset}_{idx % 5}'))
                    audio = row['audio']
                    p = f'data/vctk_hf/{subset}_{idx:04d}.wav'
                    if not os.path.exists(p):
                        sf.write(p, np.array(audio['array'], dtype=np.float32), audio['sampling_rate'])
                    manifest.append({
                        'path': p,
                        'label': cls,
                        'speaker': spk,
                        'class_idx': CLASSES.index(cls)
                    })
                    speaker_counts[cls] += 1
                    cnt += 1
                    if cnt >= 250:
                        break
                print(f'  ✓ {subset} -> {cls}: {cnt} samples')
            except Exception as se:
                print(f'  Could not load {subset}: {se}')
        with open(hf_cache_meta, 'w', encoding='utf-8') as f:
            json.dump(manifest, f, indent=2)

# Add Irish Supplement from Step 5
if 'cv_extras' in globals() and cv_extras:
    existing_irish_paths = {m['path'] for m in manifest if m['label'] == 'Irish'}
    new_cv = [c for c in cv_extras if c['path'] not in existing_irish_paths]
    manifest.extend(new_cv)
    for item in new_cv:
        speaker_counts['Irish'] += 1
    print(f'\n+ Added {len(new_cv)} Irish samples from Step 5')

print(f'\n✓ Total audio files in manifest: {len(manifest)}')
print('\nClass distribution:')
for cls in CLASSES:
    print(f'  {cls:<15}: {speaker_counts[cls]:5d}')

train_m, val_m, test_m = speaker_disjoint_splits(manifest)
print(f'\nSplits → Train: {len(train_m)} | Val: {len(val_m)} | Test: {len(test_m)}')
os.makedirs('data/manifests', exist_ok=True)
save_manifest(train_m, 'data/manifests/train.json')
save_manifest(val_m,   'data/manifests/val.json')
save_manifest(test_m,  'data/manifests/test.json')
print('✓ Manifests saved to data/manifests/')


In [ ]:
## Step 7: Training (6-class model — AMD Radeon 860M DirectML Optimized)
# Uses batch_size=2 + grad_accum=4 (effective batch=8) to fit safely in 512MB VRAM.
import os, sys, json, torch
from collections import Counter
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, balanced_accuracy_score
from tqdm.auto import tqdm
from src.data.vctk_dataset import UKAccentDataset, collate_pad, load_manifest
from src.models.wavlm_classifier import WavLMAccentClassifier

# Auto-detect DirectML (AMD Radeon 860M) -> CUDA -> CPU
DEVICE = None
try:
    import torch_directml
    if torch_directml.is_available():
        DEVICE = torch_directml.device()
except ImportError:
    pass
if DEVICE is None:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASSES    = ('RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands', 'Irish')
N          = len(CLASSES)  # 6
EPOCHS     = 15
BATCH      = 2   # Physical batch size (safe for 512MB VRAM on Radeon 860M)
GRAD_ACCUM = 4   # Effective batch size = 2 * 4 = 8
LR         = 3e-4

print(f'Training {N}-class model: {CLASSES}')
print(f'Active Device: {DEVICE}')
print(f'Batch Size: {BATCH} (Grad Accum: {GRAD_ACCUM} -> Effective Batch: {BATCH * GRAD_ACCUM})\n')

# Self-healing manifest check: ensure all labels and indices strictly match current 6 CLASSES
for split_name in ['train', 'val', 'test']:
    split_path = f'data/manifests/{split_name}.json'
    if os.path.exists(split_path):
        with open(split_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        modified = False
        valid_data = []
        for e in data:
            if e['label'] not in CLASSES:
                modified = True
                continue
            correct_idx = CLASSES.index(e['label'])
            if e['class_idx'] != correct_idx:
                e['class_idx'] = correct_idx
                modified = True
            valid_data.append(e)
        if modified:
            print(f'✓ Sanitized and re-indexed {split_path} to match {N} classes ({len(valid_data)} samples)')
            with open(split_path, 'w', encoding='utf-8') as f:
                json.dump(valid_data, f, indent=2)

# Load datasets (num_workers=0 is required for DirectML on Windows)
tr_ds = UKAccentDataset(load_manifest('data/manifests/train.json'), augment=True)
va_ds = UKAccentDataset(load_manifest('data/manifests/val.json'),   augment=False)
tr_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  collate_fn=collate_pad, num_workers=0)
va_dl = DataLoader(va_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad, num_workers=0)
print(f'Train: {len(tr_ds)} samples ({len(tr_dl)} batches)')
print(f'Val:   {len(va_ds)} samples ({len(va_dl)} batches)\n')

# Class-weighted loss to handle imbalance
cnt = Counter(e['class_idx'] for e in load_manifest('data/manifests/train.json'))
w   = torch.tensor([1.0 / max(cnt.get(i, 1), 1) for i in range(N)], dtype=torch.float32)
w   = w / w.sum() * N
criterion = torch.nn.CrossEntropyLoss(weight=w.to(DEVICE))

print('Class distribution in training set:')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:<14}: {cnt.get(i, 0):6d} (weight: {w[i]:.3f})')
print()

# Load model
model = WavLMAccentClassifier(
    pretrained_model_name='microsoft/wavlm-base-plus',
    num_classes=N,
    freeze_encoder=True,
    unfreeze_top_k_layers=2,
    dropout_p=0.3,
).to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable parameters: %d / %d (%.1f%%)\n' % (trainable, total, 100*trainable/total))

# Optimizer & scheduler
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
total_steps = max(1, (len(tr_dl) // GRAD_ACCUM) * EPOCHS)
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, total_steps // 10), total_steps)

best = 0.0
os.makedirs('checkpoints', exist_ok=True)
print(f"{'Ep':<4} {'Loss':<10} {'TrF1':<10} {'ValF1':<10} {'BalAcc':<10} Status")
print('-' * 60)

for ep in range(1, EPOCHS + 1):
    model.train()
    el, pp, ll = 0.0, [], []
    optimizer.zero_grad()
    pbar = tqdm(tr_dl, desc=f'Epoch {ep}/{EPOCHS}', leave=False)
    for step, (wav, lb, mask) in enumerate(pbar):
        wav, lb, mask = wav.to(DEVICE), lb.to(DEVICE), mask.to(DEVICE)
        out  = model(wav, attention_mask=mask)
        loss = criterion(out['logits'], lb) / GRAD_ACCUM
        loss.backward()
        el += loss.item() * GRAD_ACCUM
        pp.extend(out['logits'].argmax(-1).cpu().tolist())
        ll.extend(lb.cpu().tolist())

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(tr_dl):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
        pbar.set_postfix(loss=f'{el / (step + 1):.4f}')
    pbar.close()

    tr_f1 = f1_score(ll, pp, average='macro', zero_division=0)

    # Validation loop
    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for wav, lb, mask in va_dl:
            out = model(wav.to(DEVICE), attention_mask=mask.to(DEVICE))
            vp.extend(out['logits'].argmax(-1).cpu().tolist())
            vl.extend(lb.tolist())
    val_f1 = f1_score(vl, vp, average='macro', zero_division=0)
    bal_acc = balanced_accuracy_score(vl, vp)

    status = ''
    if val_f1 > best:
        best = val_f1
        # Save CPU tensors so checkpoint loads cleanly on any device (DirectML, CUDA, or CPU)
        cpu_state = {k: v.cpu() for k, v in model.state_dict().items()}
        torch.save(cpu_state, 'checkpoints/best_wavlm_accentsense.pt')
        status = '✓ BEST'

    print(f'{ep:<4} {el/len(tr_dl):<10.4f} {tr_f1:<10.4f} {val_f1:<10.4f} {bal_acc:<10.4f} {status}')

print('-' * 60)
print(f'✓ Training complete. Best Val Macro-F1: {best:.4f}')
print('  Saved checkpoint: checkpoints/best_wavlm_accentsense.pt')


In [ ]:
# Step 8: Test-set evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Load onto CPU first then move to DirectML device (avoids DirectML storage resize issue in torch.load)
state_dict = torch.load('checkpoints/best_wavlm_accentsense.pt', map_location='cpu')
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

te_ds = UKAccentDataset(load_manifest('data/manifests/test.json'), augment=False)
te_dl = DataLoader(te_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad, num_workers=0)

tp, tl = [], []
with torch.no_grad():
    for wav, lb, mask in te_dl:
        out = model(wav.to(DEVICE), attention_mask=mask.to(DEVICE))
        tp.extend(out['logits'].argmax(-1).cpu().tolist())
        tl.extend(lb.tolist())

print('=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(classification_report(tl, tp, labels=list(range(len(CLASSES))), target_names=list(CLASSES), zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(tl, tp, labels=list(range(len(CLASSES)))))
test_f1 = f1_score(tl, tp, average='macro', zero_division=0)
print('Final Test Macro-F1: %.4f' % test_f1)


In [ ]:
# Step 9: Verify local checkpoint for FastAPI Backend
import os

ckpt = os.path.abspath('checkpoints/best_wavlm_accentsense.pt')
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / (1024 * 1024)
    print(f'✓ Checkpoint ready: {ckpt} ({size:.1f} MB)')
    print('\nSince you trained locally, the checkpoint is ALREADY in Backend/checkpoints/!')
    print('Start the backend server with:')
    print('  uvicorn src.api.main:app --reload')
else:
    print('⚠ Checkpoint not found. Run Step 7 first.')


## After Local Training

1. Your trained checkpoint is automatically saved at `Backend/checkpoints/best_wavlm_accentsense.pt`
2. Start the backend server from `Backend/`:
   ```powershell
   .\Backend\.venv\Scripts\python.exe -m uvicorn src.api.main:app --reload
   ```
3. Verify `GET /health` → `model_loaded: true`, `supported_classes: ["RP", "Scottish", "Welsh", "Northern", "West_Midlands", "Irish"]`
